# Prompt Guard Fine-tuning (Kaggle)

Notebook nay train model `meta-llama/Llama-Prompt-Guard-2-86M` tu file du lieu local da tach rieng:
- `pyrit_test_prompts.jsonl` (chi de test PyRIT, KHONG train)
- `prompt_guard_train.jsonl` (chi de train Prompt Guard)

In [ ]:
!pip install -q -U transformers datasets accelerate sentencepiece

In [ ]:
from collections import Counter
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [ ]:
MODEL_NAME = 'meta-llama/Llama-Prompt-Guard-2-86M'
MAX_LENGTH = 512
EPOCHS = 2

DATA_CANDIDATES = [
    Path('/kaggle/input/project2-data/prompt_guard_train.jsonl'),
    Path('/kaggle/input/data/prompt_guard_train.jsonl'),
    Path('/kaggle/working/data/prompt_guard_train.jsonl'),
]

TRAIN_FILE = next((p for p in DATA_CANDIDATES if p.exists()), None)
if TRAIN_FILE is None:
    raise FileNotFoundError('Khong tim thay prompt_guard_train.jsonl. Hay upload dataset vao Kaggle Input.')

OUTPUT_DIR = Path('/kaggle/working/finetuned-prompt-guard')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_FILE

In [ ]:
dataset = Dataset.from_json(str(TRAIN_FILE))
dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'SAFE', 1: 'INJECTION'},
    label2id={'SAFE': 0, 'INJECTION': 1},
)

class_counts = Counter(int(x) for x in dataset['label'])
num_safe = class_counts.get(0, 0)
num_injection = class_counts.get(1, 0)
total_samples = num_safe + num_injection

if total_samples == 0:
    raise ValueError('Dataset rong hoac khong co cot label hop le.')

# Balanced weight: total/(num_classes * class_count)
w_safe = total_samples / (2 * max(num_safe, 1))
w_injection = total_samples / (2 * max(num_injection, 1))
class_weights = torch.tensor([w_safe, w_injection], dtype=torch.float32)

print(f'Train label distribution -> SAFE(0): {num_safe}, INJECTION(1): {num_injection}')
print(f'Class weights -> SAFE(0): {w_safe:.4f}, INJECTION(1): {w_injection:.4f}')

def preprocess(batch):
    tokenized = tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
    tokenized['labels'] = [int(x) for x in batch['label']]
    return tokenized

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset.column_names)
eval_size = min(2000, max(500, len(tokenized) // 20))
splits = tokenized.train_test_split(test_size=eval_size, seed=42) if len(tokenized) > (eval_size + 100) else {'train': tokenized, 'test': None}

splits['train'], splits['test']

In [ ]:
training_args = TrainingArguments(
    output_dir='/kaggle/working/training-output',
    num_train_epochs=EPOCHS,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    fp16=torch.cuda.is_available(),
    evaluation_strategy='epoch' if splits['test'] is not None else 'no',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=50,
    report_to='none',
)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs['labels'].long()
        model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}
        outputs = model(**model_inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=splits['train'],
    eval_dataset=splits['test'],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved model to: {OUTPUT_DIR}')

In [ ]:
!cd /kaggle/working && zip -r finetuned-prompt-guard.zip finetuned-prompt-guard > /dev/null
print('Created /kaggle/working/finetuned-prompt-guard.zip')